In [216]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
os.environ["CUDA_VISIBLE_DEVICES"] = ""


In [217]:
import torch
torch.cuda.empty_cache()

In [218]:
import deepxde as dde
dde.config.set_default_autodiff("zcs")
import numpy as np
import matplotlib.pyplot as plt
import torch

Set the default automatic differentiation to zcs mode.


In [219]:
spatial_geom = dde.geometry.Rectangle([0.0, 0.0], [1.0, 1.0])
timedomain = dde.geometry.TimeDomain(0.0, 1.0)
geom = dde.geometry.GeometryXTime(spatial_geom, timedomain)

THE nu in the equation can either be a trainable parameter and be passed as a dde.Variable or it can be a variable parameter set during inference(part of the inputs).

In [220]:
pho = 1

In [221]:
def Navier_stokes(x, y, z): #x[1] is the trunk inputs, x[0] is the branch inputs(x is a nested list of both branch and trunk inputs)
    trunk_coords = x
    branch_coords = x[0]

    u = y[:, 0:1]
    v = y[:, 1:2]
    p = y[:, 2:3]

    nu = 0.1

    #continuity equation
    du_dx = dde.grad.jacobian(u, trunk_coords, i=0, j=0)
    dv_dy = dde.grad.jacobian(v, trunk_coords, i=1, j=1)

    #x-momentum equation
    du_dt = dde.grad.jacobian(u, trunk_coords, i=0, j=2)
    du_dy = dde.grad.jacobian(u, trunk_coords, i=0, j=1)
    dp_dx = dde.grad.jacobian(p, trunk_coords, i=2, j=0)
    du_xx = dde.grad.hessian(u, trunk_coords, component=0, i=0, j=0)
    du_yy = dde.grad.hessian(u, trunk_coords, component=0, i=1, j=1)
    x_mom = du_dt + u*du_dx + v*du_dy + (1/pho)*dp_dx - nu*(du_xx + du_yy)

    #y-momentum equation
    dv_dt = dde.grad.jacobian(v, trunk_coords, i=1, j=2)
    dv_dy = dde.grad.jacobian(v, trunk_coords, i=1, j=1)
    dp_dy = dde.grad.jacobian(p, trunk_coords, i=2, j=1)
    dv_xx = dde.grad.hessian(v, trunk_coords, component=0, i=0, j=0)
    dv_yy = dde.grad.hessian(v, trunk_coords, component=0, i=1, j=1)
    dv_dx = dde.grad.jacobian(v, trunk_coords, i=1, j=0)
    y_mom = dv_dt + u*dv_dx + v*dv_dy + (1/pho)*dp_dy - nu*(dv_xx + dv_yy)

    return [x_mom, y_mom]

#enforce the stream function using an output_transform


In [222]:
#Enforicing the stream function psi here to satisfy the continuity automatically

def output_transform(inputs, outputs):
    trunk_inputs = inputs[1]
    x = trunk_inputs[:, 0:1]
    y = trunk_inputs[:, 1:2]
    t = trunk_inputs[:, 2:3]

    psi = outputs[:, 0:1]
    p = outputs[:, 1:2]

    psi = x * (1 - x) * y * (1 - y) * psi #hard boundary(enforcing 0.0 output at x = 0.0, y=0.0)

    u = dde.grad.jacobian(psi, trunk_inputs, i=0, j=1)
    v = - dde.grad.jacobian(psi, trunk_inputs, i=0, j=0)

    return torch.cat([u, v, p], dim=1)


In [223]:
def is_on_right_boundary(X, on_boundary):
    if not on_boundary:
        return False

    return dde.utils.isclose(X[0], 1.0)

def pressure_outlet(X):
    return 0.0


def is_on_wall(X, on_boundary):
    if not on_boundary:
        return False

    return dde.utils.isclose(X[1], 0.0) or dde.utils.isclose(X[1], 1.0)

def wall_condition(X):
    return 0.0

In [224]:
bc_p = dde.icbc.DirichletBC(geom, pressure_outlet, is_on_right_boundary, component=2)
bc_u = dde.icbc.DirichletBC(geom, wall_condition, is_on_wall, component=0)
bc_v = dde.icbc.DirichletBC(geom, wall_condition, is_on_wall, component=1)

In [225]:
data = dde.data.TimePDE(
    geom,
    Navier_stokes,
    [bc_u, bc_v, bc_p],
    num_domain=300,
    num_boundary=50,
    num_test=200,
    train_distribution="pseudo"
)

DEFINING THE OPERATOR AND FUNCTION SPACE

In [226]:
#Defining the function space
func_space = dde.data.GRF(length_scale=0.2)

n_sensors = 50

eval_points = np.linspace(0, 1, n_sensors)[:, None]


In [227]:
pde_op = dde.data.PDEOperator(
    pde=data,
    function_space=func_space,
    evaluation_points=eval_points,
    num_function=50,
    num_test=10,
    function_variables=[0],
)

In [228]:
#Model definition
n_sensors = 50

net = dde.nn.DeepONet(
    [n_sensors]+[64]*2, # n_sensors + 1 to add reynolds number as a variable input feature(not added yet)
    [3]+[64]*2,
    "tanh",
    "Glorot normal",
    num_outputs=2
)
net.apply_output_transform(output_transform)

In [229]:
model = dde.Model(pde_op, net)

model.compile("adam", lr=0.001)

Compiling model...
'compile' took 0.000253 s



In [230]:
early_stop = dde.callbacks.EarlyStopping(patience=5)
out_dir = "pino_models"
os.makedirs(out_dir, exist_ok=True)
ckpt = dde.callbacks.ModelCheckpoint(out_dir)

In [231]:
resampler = dde.callbacks.PDEPointResampler(period=1000)

In [232]:
losshistory, train_state = model.train(iterations=15000, callbacks=[early_stop], disregard_previous_best=True,
                                       display_every=1000)
dir = "pino_01"
os.makedirs(dir, exist_ok=True)
dde.saveplot(losshistory, train_state, issave=True, isplot=True, output_dir=dir)

Training model...



[W822 20:14:21.087823065 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 20971520 bytes (free: 27983872, total: 6076825600).
[W822 20:14:21.087999152 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 20971520 bytes (free: 27983872, total: 6076825600).


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 5.66 GiB of which 26.69 MiB is free. Including non-PyTorch memory, this process has 5.21 GiB memory in use. Of the allocated memory 4.95 GiB is allocated by PyTorch, and 151.19 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [182]:
dde.optimizers.config.set_LBFGS_options(maxiter=30000)
model.compile("L-BFGS")
losshistory, train_state = model.train(callbacks=[early_stop, ckpt], disregard_previous_best=True, display_every=1000)

dir = "pino_01_finetuned"
os.makedirs(dir, exist_ok=True)
dde.saveplot(losshistory, train_state, issave=True, isplot=True, output_dir=dir)

Compiling model...
'compile' took 0.000579 s

Training model...



[W822 20:02:09.178928094 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 20971520 bytes (free: 41811968, total: 6076825600).
[W822 20:02:09.179073059 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 20971520 bytes (free: 41811968, total: 6076825600).


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 5.66 GiB of which 39.88 MiB is free. Including non-PyTorch memory, this process has 5.19 GiB memory in use. Of the allocated memory 4.93 GiB is allocated by PyTorch, and 151.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

EVAL

In [ ]:
test_x = np.random.uniform(0.0, 1.0, (5000, 1))
test_y = np.random.uniform(0.0, 0.1, (5000, 1))
test_t = np.random.uniform(0.0, 0.1, (5000, 1))
test_xyt = np.hstack([test_x, test_y, test_t])

residuals = model.predict(test_xyt, operator=Navier_stokes)

print(f"X_mom: {residuals[0]}")
print(f"Y_mom: {residuals[1]}")
